# Mixture-of-experts benchmark: LoRA on every expert, on one GPU

<small>Copyright Alejandro Martínez Corriá and the Thinkube contributors. SPDX-License-Identifier: Apache-2.0</small>

This notebook trains LoRA on a mixture-of-experts model, its routed experts included, and compares three ways of computing it:

| Arm | What it is |
|---|---|
| `meepsquared` | meepsquared with its defaults: each expert's adapter computed on the tokens routed to that expert, the frozen weights in BF16 |
| `meepsquared-fp8` | the same, with the frozen weights held as FP8 (`load_in_fp8=True`), the expert stacks included: half the memory of the experts |
| `meepsquared-peft-experts` | the experts computed as PEFT computes them: the full weight `W + B·A` of every expert built at every pass |

Every arm trains LoRA rank 16 (alpha 16, no dropout) on the attention, the gated DeltaNet projections, the shared expert and every routed expert, with 8-bit AdamW, a constant learning rate of 1e-4 and BF16, on the same no_robots conversations in the same order, the loss on the answers only. Each run is one call of `moe.py`, next to this notebook, and appends one line to `results/moe.jsonl` with its settings, its environment and its numbers.

## Settings

`MODEL_SOURCE` says where the model comes from:

* `"huggingface"`: downloaded from the Hugging Face Hub by its id.
* `"thinkube"`: on [Thinkube](https://thinkube.org), from the cluster's model mirror. A mirrored model was copied once into the cluster's storage, which every GPU node reads, so no run downloads anything: for a 70 GB model, that is the difference between minutes and an hour. Mirror a model in Thinkube Control (AI > Models) before you use it.

A notebook job can override any of these with its parameters.

In [ ]:
PARAMETERS = globals().get("PARAMETERS", {})

MODEL_SOURCE = PARAMETERS.get("MODEL_SOURCE", "huggingface")      # "thinkube" or "huggingface"
MODEL = PARAMETERS.get("MODEL", "Qwen/Qwen3.6-35B-A3B")
ARMS = PARAMETERS.get("ARMS", ["meepsquared", "meepsquared-fp8", "meepsquared-peft-experts"])
STEPS = PARAMETERS.get("STEPS", 30)
WARMUP = PARAMETERS.get("WARMUP", 10)   # the first minutes of a run are irregular; see below
BATCH = PARAMETERS.get("BATCH", 32)           # conversations per step
# meepsquared's gradient checkpointing: auto, true, false, a number of layers or a fraction
CHECKPOINTING = PARAMETERS.get("CHECKPOINTING", "auto")
# The benchmarks folder of meepsquared: this notebook's folder, unless a job runs a copy elsewhere
BENCHMARKS = PARAMETERS.get("BENCHMARKS")

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

BENCHMARKS = Path(BENCHMARKS) if BENCHMARKS else Path.cwd()
if not (BENCHMARKS / "moe.py").is_file():
    raise FileNotFoundError(f"moe.py is not in {BENCHMARKS}: run this notebook from the benchmarks folder of meepsquared, "
                            "or set BENCHMARKS to that folder")
RESULTS = BENCHMARKS / "results" / "moe.jsonl"

sys.path.insert(0, str(BENCHMARKS))
from common import environment

print(json.dumps(environment(), indent=1))

## Where the model comes from

On Thinkube, `thinkube_models.mirrored_model_path` gives the folder of the mirrored model, and stops with an error naming it if it is not mirrored, before any training starts.

In [ ]:
if MODEL_SOURCE == "thinkube":
    import thinkube_models as tkm

    print(f"{MODEL}: {tkm.mirrored_model_path(MODEL)}")
else:
    print("The model is downloaded from the Hugging Face Hub")

## Run

Each run prints its steps as it goes. A run that fails stops the notebook with its own error.

The first ten steps of a run are not measured. In a run's first minutes some steps take two to five times a steady step: the kernels compile, and on a machine whose memory the GPU shares, the system is still caching the model's files it just read. The Triton kernels a run compiles are kept in a cache folder next to the results.

In [ ]:
CACHE = BENCHMARKS / ".triton-cache"


def run(arm, steps=STEPS, warmup=WARMUP, results=RESULTS):
    script = [str(BENCHMARKS / "moe.py"), "--model", MODEL, "--source", MODEL_SOURCE, "--steps", str(steps),
              "--warmup", str(warmup), "--batch", str(BATCH), "--checkpointing", str(CHECKPOINTING),
              "--results", str(results), "--label", arm]
    if arm.startswith("meepsquared-fp8"):
        script.append("--fp8")
    elif arm == "meepsquared-peft-experts":
        script.append("--peft-experts")
    print(f"=== {MODEL} {arm}", flush=True)
    process = subprocess.Popen([sys.executable, "-u", *script], cwd=BENCHMARKS, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True,
                               env={**os.environ, "TOKENIZERS_PARALLELISM": "false", "TRITON_CACHE_DIR": str(CACHE)})
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"{MODEL} {arm} failed with exit code {process.returncode}")


for arm in ARMS:
    run(arm)

## Results

Seconds per step and tokens per second count from step `WARMUP` to the last; the first ten steps of a run are irregular and left out. The peak memory is what the GPU held. The losses show whether the arms trained the same tokens: the FP8 arm's differ slightly, as the quantization of the frozen weights does.

In [ ]:
import pandas as pd

rows = [json.loads(line) for line in RESULTS.read_text().splitlines()]
table = pd.DataFrame([r for r in rows if r["model"] == MODEL and r.get("source") == MODEL_SOURCE and r["batch"] == BATCH])
table = table.drop_duplicates(["arm"], keep="last").set_index("arm")
table["peak_gib"] = table["peak_gib"].map(max)
display(table.loc[[a for a in ARMS if a in table.index],
                  ["s_per_step", "steps_per_hour", "tokens_per_s", "peak_gib", "checkpointing", "checkpointed_layers", "trainable_m"]])
for arm, losses in table["losses"].items():
    print(f"{arm}: first {losses[0]}, last {losses[-1]}")